# धडा 08 - मल्टी-एजंट डिझाइन पॅटर्न


## सेटअप


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## मल्टी-एजंट सिस्टम का?

खर्‍या जगातील कामांमध्ये ट्रिप नियोजन सारख्या अनेक वेगवेगळ्या कौशल्यांची गरज असते — लॉजिस्टिक, स्थानिक ज्ञान, बजेटिंग, आणि बरेच काही. सगळं हाताळणारा एकच एजंट लवकरच अव्यवस्थित होऊन जातो.

मल्टी-एजंट सिस्टम्स याला **विशेषीकरण** द्वारे सोडवतात: प्रत्येक एजंट एका विशिष्ट क्षेत्रावर लक्ष केंद्रित करतो, ज्यामुळे सामान्यत: पेक्षा अधिक दर्जेदार परिणाम मिळतात. ते **विस्तारक्षमतेत** सुधारणा करतात — तुम्ही नवीन एजंट जोडू शकता (उदा. फ्लाइट विशेषज्ञ, रेस्टॉरंट समीक्षक) विद्यमान वर्कफ्लो पुन्हा लिहिण्याशिवाय. एजंट एकमेकांशी संरचित पाइपलाइनद्वारे जोडले जातात, ज्यात एकाकडून दुसऱ्याकडे संदर्भ पास केला जातो.


## विशेषज्ञ एजंट तयार करणे


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## अनुक्रमिक वर्कफ्लो तयार करणे

`WorkflowBuilder` आपल्याला एजंट्सना निर्देशित ग्राफमध्ये वायर करण्याची परवानगी देते. येथे आपण एक सोपा दोन-टप्यांचा पाइपलाइन तयार करतो: **TravelPlanner** प्रवासाचा आराखडा तयार करतो, नंतर **TravelConcierge** त्याचे पुनरवलोकन करून त्याला सुधारणा करते.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## कार्यप्रवाहात आणखी एजंट्स जोडणे

मल्टि-एजंट पॅटर्नचा एक मोठा फायदा म्हणजे त्याचा विस्तार करणे सोपे असते. खाली आपण एक **BudgetReviewer** एजंट जोडतो जो प्रवाश्याच्या बजेटशी योजना तपासतो, अशा वस्तू ओळखतो ज्यामुळे खर्च मर्यादेपेक्षा जास्त होऊ शकतो, आणि पैसे वाचवण्यासाठी पर्याय सुचवतो. कार्यप्रवाह आता सलग तीन एजंट्स चालवतो:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## सारांश

या धड्यामध्ये तुम्ही हे शिकलात कसे करायचे:

1. **विशेषीकृत एजंट निर्माण करणे** — प्रत्येक एजंटचा एक लक्षित भूमिका असते (योजना, कन्सियर्ज, बजेट पुनरावलोकन).
2. **एजंट्सना अनुक्रमिक वर्कफ्लोमध्ये जोडणे** `WorkflowBuilder` आणि `add_edge` वापरून.
3. **मल्टी-एजंट पाईपलाइनमधून आउटपुट स्ट्रिम करणे**, कोणता एजंट बोलत आहे हे ट्रॅक करत.
4. **वर्कफ्लो विस्तारणे** नवीन एजंट्स साखळीत जोडून विद्यमान एजंट्समध्ये बदल न करता.

मल्टी-एजंट डिझाइन पॅटर्न प्रत्येक एजंट सोपा ठेवतो, तरीही एकट्या एजंटपेक्षा अधिक समृद्ध, अधिक सखोलपणे पुनरावलोकन केलेले परिणाम तयार करतो.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**कार्यालयीन सूचना**:  
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्नशील आहोत, परंतु कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये चुका किंवा अपूर्णता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत एकाधिकृत स्रोत मानला पाहिजे. महत्त्वाच्या माहितीसाठी व्यावसायिक मानवी अनुवाद शिफारस केला जातो. या अनुवादाच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थांच्या जबाबदारी आम्ही घेत नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
